# Exercise 5 — Strategy Backtest Comparison

Run all four strategies through the backtester and compare their metrics. The goal is not to find the 'best' strategy on synthetic data — sine-wave prices are not real. The goal is to practise reading the metrics table and understanding the trade-offs between trend-following, mean-reversion, momentum, and combined strategies.

In [ ]:
import pandas as pd, math, warnings

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

def _sma(s, w):  return s.rolling(w).mean()
def _ema(s, w):  return s.ewm(span=w, adjust=False).mean()
def _rsi(s, w):
    d = s.diff()
    g = d.clip(lower=0).rolling(w).mean()
    l = (-d.clip(upper=0)).rolling(w).mean()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rs = g / l
    return 100 - (100 / (1 + rs))
def sma_crossover(df, fast=20, slow=50):
    close = df["Close"]
    return (_sma(close, fast) > _sma(close, slow)).fillna(False).astype(int)
def rsi_mean_reversion(df, window=14, oversold=30, overbought=70):
    rsi_s  = _rsi(df["Close"], window)
    signal = pd.Series(float("nan"), index=df.index)
    signal[rsi_s < oversold]   = 1.0
    signal[rsi_s > overbought] = 0.0
    return signal.ffill().fillna(0).astype(int)
def macd_cross(df, fast=12, slow=26, signal=9):
    close       = df["Close"]
    macd_line   = _ema(close, fast) - _ema(close, slow)
    signal_line = _ema(macd_line, signal)
    return (macd_line > signal_line).astype(int)
def combined_signal(df, fast=20, slow=50, macd_fast=12, macd_slow=26, macd_sig=9):
    sma_sig  = sma_crossover(df, fast, slow)
    macd_sig = macd_cross(df, macd_fast, macd_slow, macd_sig)
    return ((sma_sig == 1) & (macd_sig == 1)).astype(int)
def _compute_returns(df):  return df["Close"].pct_change()
def _compute_equity(r):    return (1 + r.fillna(0)).cumprod()
def _max_dd(eq):
    peak = eq.cummax()
    return float(((eq - peak) / peak).min())
def _sharpe(r):
    c = r.dropna()
    if len(c) == 0 or c.std() == 0: return 0.0
    return float(c.mean() / c.std() * (252 ** 0.5))
def run_backtest(df, signals, label="strategy"):
    mr  = _compute_returns(df)
    pos = signals.shift(1).fillna(0)
    sr  = pos * mr
    eq  = _compute_equity(sr)
    c   = sr.dropna(); n = len(c)
    tr  = float(eq.iloc[-1] - 1.0)
    base = 1.0 + tr
    ar  = float(base ** (252.0 / max(n, 1)) - 1) if base > 0 else -1.0
    pos_diff = pos.diff().fillna(0)
    return {
        "label":             label,
        "total_return":      tr,
        "annualized_return": ar,
        "sharpe_ratio":      _sharpe(sr),
        "max_drawdown":      _max_dd(eq),
        "win_rate":          float((c > 0).sum() / max(n, 1)),
        "n_trades":          int((pos_diff != 0).sum()),
        "equity":            eq,
    }


### Build signals and run backtests

In [ ]:
df = _synthetic(n=252)

sig_sma  = sma_crossover(df)
sig_rsi  = rsi_mean_reversion(df, oversold=35, overbought=65)
sig_macd = macd_cross(df)
sig_comb = combined_signal(df)
sig_bah  = pd.Series(1, index=df.index)   # benchmark: buy-and-hold

r_sma  = run_backtest(df, sig_sma,  "SMA-cross")
r_rsi  = run_backtest(df, sig_rsi,  "RSI-MR")
r_macd = run_backtest(df, sig_macd, "MACD-cross")
r_comb = run_backtest(df, sig_comb, "Combined")
r_bah  = run_backtest(df, sig_bah,  "Buy-Hold")


### Print metrics table

In [ ]:
checks = 0

# 1 — all results have required keys
try:
    REQUIRED = {"total_return","annualized_return","sharpe_ratio","max_drawdown","win_rate","n_trades","equity"}
    for r in [r_sma, r_rsi, r_macd, r_comb, r_bah]:
        assert REQUIRED.issubset(r.keys()), f"missing keys in {r.get('label')}"
    checks += 1; print("✅ 1 all backtest results have required keys")
except Exception as e:
    print("❌ 1:", e)

# 2 — equity[-1] == 1 + total_return for all
try:
    for r in [r_sma, r_rsi, r_macd, r_comb, r_bah]:
        diff = abs(r["equity"].iloc[-1] - (1 + r["total_return"]))
        assert diff < 1e-9, f"equity/total_return mismatch for {r['label']}"
    checks += 1; print("✅ 2 equity[-1] == 1 + total_return for all strategies")
except Exception as e:
    print("❌ 2:", e)

# 3 — combined has fewer or equal trades than SMA crossover
try:
    assert r_comb["n_trades"] <= r_sma["n_trades"] + 1,         f"combined ({r_comb['n_trades']}) has more trades than SMA ({r_sma['n_trades']})"
    checks += 1; print("✅ 3 combined strategy has fewer or equal trades")
except Exception as e:
    print("❌ 3:", e)

# 4 — max_drawdown is negative for all strategies
try:
    for r in [r_sma, r_rsi, r_macd, r_comb, r_bah]:
        assert r["max_drawdown"] <= 1e-9,             f"{r['label']}: drawdown should be ≤ 0, got {r['max_drawdown']}"
    checks += 1; print("✅ 4 all max_drawdown values are ≤ 0")
except Exception as e:
    print("❌ 4:", e)

# 5 — print comparison table
try:
    results = [r_bah, r_sma, r_rsi, r_macd, r_comb]
    print(f"\n{'Metric':<20}", end="")
    for r in results: print(f" {r['label']:>10}", end="")
    print("\n" + "-" * 72)
    for key, fmt in [
        ("total_return",      ".2%"),
        ("sharpe_ratio",      ".3f"),
        ("max_drawdown",      ".2%"),
        ("win_rate",          ".2%"),
        ("n_trades",          "d"),
    ]:
        print(f"{key:<20}", end="")
        for r in results:
            v = r[key]
            if fmt == "d":
                print(f" {v:>10d}", end="")
            else:
                print(f" {v:>{10}{fmt}}", end="")
        print()
    checks += 1; print("\n✅ 5 comparison table printed")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
